# Neural follow-up (optional Colab GPU)

Run on a GPU runtime. Upload the original dissertation workbook when prompted; no Bloomberg ZIP is needed. This notebook reconstructs the audited panel, tunes on the first purged training/validation fold, and trains both neural models. Download the single results ZIP at the end. Runtime depends on GPU and trial configuration. Do not mix results from different runs.


In [ ]:
%pip install -q optuna scikit-learn openpyxl pyarrow


In [ ]:
from pathlib import Path
import os, subprocess, zipfile
root = Path("/content/anna_followup")
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
files = {'experiments/putwriting/followup/neural_src/models.py': '"""Modelos neurais diretos e retomáveis para os 22 horizontes."""\n\nfrom __future__ import annotations\n\nimport copy\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\nfrom utils import qlike_by_horizon, set_seed\n\n\ndef _build_tsmixer(input_size: int, n_features: int, horizon: int,\n                   n_block: int, ff_dim: int, dropout: float):\n    import torch\n    from torch import nn\n\n    class MixerBlock(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.time_norm = nn.LayerNorm(input_size)\n            self.time_mlp = nn.Sequential(\n                nn.Linear(input_size, ff_dim), nn.GELU(), nn.Dropout(dropout),\n                nn.Linear(ff_dim, input_size),\n            )\n            self.feature_norm = nn.LayerNorm(n_features)\n            self.feature_mlp = nn.Sequential(\n                nn.Linear(n_features, ff_dim), nn.GELU(), nn.Dropout(dropout),\n                nn.Linear(ff_dim, n_features),\n            )\n\n        def forward(self, x):\n            z = x.transpose(1, 2)\n            z = z + self.time_mlp(self.time_norm(z))\n            z = z.transpose(1, 2)\n            return z + self.feature_mlp(self.feature_norm(z))\n\n    class DirectTSMixer(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.blocks = nn.Sequential(*[MixerBlock() for _ in range(n_block)])\n            self.head = nn.Sequential(\n                nn.LayerNorm(n_features), nn.Flatten(),\n                nn.Linear(input_size * n_features, horizon),\n            )\n\n        def forward(self, x):\n            return self.head(self.blocks(x))\n\n    return DirectTSMixer()\n\n\ndef _build_lstm(n_features: int, horizon: int, hidden_size: int, dropout: float):\n    import torch\n    from torch import nn\n\n    class DirectLSTM(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True,\n                                num_layers=1, dropout=0.0)\n            self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Dropout(dropout),\n                                      nn.Linear(hidden_size, horizon))\n\n        def forward(self, x):\n            z, _ = self.lstm(x)\n            return self.head(z[:, -1])\n\n    return DirectLSTM()\n\n\nclass DirectVolatilityRegressor:\n    """Treinador PyTorch com log-target, early stopping e histórico."""\n\n    def __init__(self, model_type: str = "tsmixerx", input_size: int = 66,\n                 n_block: int = 3, ff_dim: int = 64, hidden_size: int = 64,\n                 dropout: float = 0.1, learning_rate: float = 1e-3,\n                 batch_size: int = 64, max_steps: int = 1000,\n                 loss: str = "huber", val_check_steps: int = 20,\n                 early_stop_patience: int = 10, seed: int = C.SEED,\n                 device: str | None = None):\n        self.model_type = model_type.lower()\n        self.input_size = int(input_size)\n        self.n_block = int(n_block)\n        self.ff_dim = int(ff_dim)\n        self.hidden_size = int(hidden_size)\n        self.dropout = float(dropout)\n        self.learning_rate = float(learning_rate)\n        self.batch_size = int(batch_size)\n        self.max_steps = int(max_steps)\n        self.loss_name = loss\n        self.val_check_steps = int(val_check_steps)\n        self.early_stop_patience = int(early_stop_patience)\n        self.seed = int(seed)\n        self.requested_device = device\n        self.model = None\n        self.history: list[dict] = []\n\n    def _build(self, n_features: int):\n        if self.model_type == "lstm":\n            return _build_lstm(n_features, C.HORIZON, self.hidden_size, self.dropout)\n        return _build_tsmixer(self.input_size, n_features, C.HORIZON,\n                              self.n_block, self.ff_dim, self.dropout)\n\n    def _fit_scalers(self, x: np.ndarray, y: np.ndarray) -> None:\n        flat = x.reshape(-1, x.shape[-1]).astype(float)\n        self.x_center = np.nanmedian(flat, axis=0)\n        self.x_center = np.where(np.isfinite(self.x_center), self.x_center, 0.0)\n        q75 = np.nanpercentile(flat, 75, axis=0)\n        q25 = np.nanpercentile(flat, 25, axis=0)\n        self.x_scale = q75 - q25\n        self.x_scale = np.where(np.isfinite(self.x_scale) & (self.x_scale > 1e-8), self.x_scale, 1.0)\n\n        log_y = np.log(np.maximum(y.astype(float), C.EPS))\n        self.y_center = np.mean(log_y, axis=0)\n        self.y_scale = np.std(log_y, axis=0)\n        self.y_scale = np.where(np.isfinite(self.y_scale) & (self.y_scale > 1e-8), self.y_scale, 1.0)\n\n    def _scale_x(self, x: np.ndarray) -> np.ndarray:\n        return np.nan_to_num((x - self.x_center) / self.x_scale,\n                             nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)\n\n    def _scale_y(self, y: np.ndarray) -> np.ndarray:\n        return ((np.log(np.maximum(y, C.EPS)) - self.y_center) / self.y_scale).astype(np.float32)\n\n    def _inverse_y(self, y_scaled: np.ndarray) -> np.ndarray:\n        return np.exp(np.clip(y_scaled * self.y_scale + self.y_center, -30, 10))\n\n    def fit(self, train_x: np.ndarray, train_y: np.ndarray,\n            val_x: np.ndarray, val_y: np.ndarray, trial=None) -> "DirectVolatilityRegressor":\n        import torch\n        from torch import nn\n\n        set_seed(self.seed)\n        if self.requested_device:\n            self.device = torch.device(self.requested_device)\n        else:\n            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self._fit_scalers(train_x, train_y)\n        tx = torch.tensor(self._scale_x(train_x), device=self.device)\n        ty = torch.tensor(self._scale_y(train_y), device=self.device)\n        vx = torch.tensor(self._scale_x(val_x), device=self.device)\n        self.model = self._build(train_x.shape[-1]).to(self.device)\n        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=1e-5)\n        if self.loss_name == "mse":\n            loss_fn = nn.MSELoss()\n        elif self.loss_name == "mae":\n            loss_fn = nn.L1Loss()\n        else:\n            loss_fn = nn.SmoothL1Loss()\n\n        generator = torch.Generator(device=self.device)\n        generator.manual_seed(self.seed)\n        best_state, best_score, bad_checks = None, np.inf, 0\n        self.history = []\n        batch_size = max(1, min(self.batch_size, len(tx)))\n        for step in range(1, self.max_steps + 1):\n            self.model.train()\n            idx = torch.randint(0, len(tx), (batch_size,), generator=generator, device=self.device)\n            optimizer.zero_grad(set_to_none=True)\n            train_loss = loss_fn(self.model(tx[idx]), ty[idx])\n            train_loss.backward()\n            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)\n            optimizer.step()\n\n            if step % self.val_check_steps != 0 and step != self.max_steps:\n                continue\n            self.model.eval()\n            with torch.no_grad():\n                scaled = self.model(vx).detach().cpu().numpy()\n            val_pred = self._inverse_y(scaled)\n            qlike_h = qlike_by_horizon(val_y, val_pred)\n            val_score = float(np.mean(qlike_h))\n            val_mse = float(np.mean((val_y - val_pred) ** 2))\n            improved = val_score < best_score - 1e-8\n            if improved:\n                best_score = val_score\n                best_state = copy.deepcopy(self.model.state_dict())\n                bad_checks = 0\n            else:\n                bad_checks += 1\n            row = {\n                "step": step,\n                "train_loss": float(train_loss.detach().cpu()),\n                "validation_qlike": val_score,\n                "validation_qlike_h22": float(qlike_h[-1]),\n                "validation_mse": val_mse,\n                "best_so_far": best_score,\n                "is_best": improved,\n                "bad_checks": bad_checks,\n                "learning_rate": optimizer.param_groups[0]["lr"],\n                "stopped_early": False,\n            }\n            self.history.append(row)\n            if trial is not None:\n                trial.report(val_score, step=step)\n                if trial.should_prune():\n                    import optuna\n\n                    raise optuna.TrialPruned(f"Pruned at step {step}")\n            if bad_checks >= self.early_stop_patience:\n                self.history[-1]["stopped_early"] = True\n                break\n        if best_state is None:\n            raise RuntimeError("Nenhum checkpoint de validação foi produzido.")\n        self.model.load_state_dict(best_state)\n        self.best_validation_qlike = best_score\n        return self\n\n    def predict(self, x: np.ndarray, batch_size: int = 512) -> np.ndarray:\n        if self.model is None:\n            raise RuntimeError("Modelo não ajustado.")\n        import torch\n\n        values = torch.tensor(self._scale_x(x), device=self.device)\n        chunks = []\n        self.model.eval()\n        with torch.no_grad():\n            for start in range(0, len(values), batch_size):\n                chunks.append(self.model(values[start:start + batch_size]).detach().cpu().numpy())\n        return np.maximum(self._inverse_y(np.concatenate(chunks)), C.EPS)\n\n    def history_frame(self, **metadata) -> pd.DataFrame:\n        out = pd.DataFrame(self.history)\n        for key, value in metadata.items():\n            out[key] = value\n        return out\n\n    def config_dict(self) -> dict:\n        return {\n            "model_type": self.model_type,\n            "input_size": self.input_size,\n            "n_block": self.n_block,\n            "ff_dim": self.ff_dim,\n            "hidden_size": self.hidden_size,\n            "dropout": self.dropout,\n            "learning_rate": self.learning_rate,\n            "batch_size": self.batch_size,\n            "max_steps": self.max_steps,\n            "loss": self.loss_name,\n            "val_check_steps": self.val_check_steps,\n            "early_stop_patience": self.early_stop_patience,\n            "seed": self.seed,\n        }\n\n    def save(self, path: Path, features: list[str], metadata: dict | None = None) -> None:\n        import torch\n\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        torch.save({\n            "format_version": 2,\n            "config": self.config_dict(),\n            "features": features,\n            "x_center": self.x_center,\n            "x_scale": self.x_scale,\n            "y_center": self.y_center,\n            "y_scale": self.y_scale,\n            "state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},\n            "history": self.history,\n            "metadata": metadata or {},\n        }, path)\n\n    @classmethod\n    def load(cls, path: Path, device: str | None = None) -> tuple["DirectVolatilityRegressor", dict]:\n        import torch\n\n        map_location = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        payload = torch.load(path, map_location=map_location, weights_only=False)\n        config = dict(payload["config"])\n        config["device"] = map_location\n        model = cls(**config)\n        model.x_center = np.asarray(payload["x_center"])\n        model.x_scale = np.asarray(payload["x_scale"])\n        model.y_center = np.asarray(payload["y_center"])\n        model.y_scale = np.asarray(payload["y_scale"])\n        model.device = torch.device(map_location)\n        model.model = model._build(len(payload["features"])).to(model.device)\n        model.model.load_state_dict(payload["state_dict"])\n        model.model.eval()\n        model.history = payload.get("history", [])\n        return model, payload\n\n', 'experiments/putwriting/followup/neural_src/config.py': '"""Configuração isolada do experimento diário H=22."""\n\nimport os\nfrom pathlib import Path\n\nEXPERIMENT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = Path(__file__).resolve().parents[3]\n\nSOURCE_XLSX = REPO_ROOT / "base de dados dissertação.xlsx"\nDATA_DIR = Path(os.environ.get("DAILY_H22_DATA_DIR", EXPERIMENT_ROOT / "data"))\nDAILY_PANEL = DATA_DIR / "daily_panel.parquet"\nSPLIT_MANIFEST = DATA_DIR / "split_manifest.csv"\n\nRESULTS_DIR = Path(os.environ.get("DAILY_H22_RESULTS_DIR", EXPERIMENT_ROOT / "results"))\nFORECAST_DIR = RESULTS_DIR / "forecasts"\nINPUTS_DIR = RESULTS_DIR / "inputs"\nMETRICS_DIR = RESULTS_DIR / "metrics"\nFIGURES_DIR = RESULTS_DIR / "figures"\nMODELS_DIR = RESULTS_DIR / "models"\nHP_DIR = RESULTS_DIR / "hp_search"\nHISTORY_DIR = RESULTS_DIR / "training_history"\nEXPLAIN_DIR = RESULTS_DIR / "explainability"\n\nTARGET = "rv_yz_22"\nHORIZON = 22\nRV_WINDOW = 22\nANNUALIZATION = 252\n\nTRAIN_YEARS = 8\nVAL_YEARS = 2\nTEST_YEARS = 1\nSTEP_YEARS = 1\n\n# Conversão dos lags mensais acordados {1,3,6,12,24} para pregões.\nINPUT_SIZES = [22, 66, 132, 264, 528]\n\nSEED = 42\nEPS = 1e-10\n\nPRICE_RETURN_PAIRS = {\n    "spx_close": "ret_spx_log",\n    "ndx_close": "ret_ndx_log",\n    "indu_close": "ret_indu_log",\n    "nky_close": "ret_nky_log",\n    "ukx_close": "ret_ukx_log",\n    "wti_close": "ret_wti_log",\n    "gold_close": "ret_gold_log",\n    "dxy": "ret_dxy_log",\n}\n\nMONTHLY_MACRO = ["industrial_production", "cpi", "unemployment"]\nWEEKLY_MACRO = ["stress_index"]\n\n# VIX3M é deliberadamente excluído da análise principal 2000-2025.\nEXCLUDED_MAIN = {\n    "vix3m",\n    "vix_term_spread",\n    "ret_nky_simple",\n    "ret_ukx_simple",\n    "ret_wti_simple",\n    "retorno_acum_22d",\n    "rv_semanal",\n    "rv_mensal",\n    "rolling_std_22d",\n}\n\nNON_FEATURE_COLUMNS = {\n    "date",\n    "unique_id",\n    TARGET,\n    "close_variance_forward_22",\n    "close_volatility_forward_22",\n}\n', 'experiments/putwriting/followup/neural_src/feature_selection.py': '"""Seleção de features ajustada exclusivamente nas origens de treino."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\nfrom utils import feature_columns, target_matrix\n\n\ndef rank_features(panel: pd.DataFrame, train_origins: np.ndarray,\n                  missing_threshold: float = 0.25,\n                  corr_threshold: float = 0.99) -> pd.DataFrame:\n    candidates = [c for c in feature_columns(panel) if c != C.TARGET]\n    x = panel.loc[train_origins, candidates].apply(pd.to_numeric, errors="coerce")\n    missing = x.isna().mean()\n    variance = x.var(ddof=0)\n    eligible = [c for c in candidates if missing[c] <= missing_threshold\n                and np.isfinite(variance[c]) and variance[c] > 1e-12]\n\n    # Remove pares quase determinísticos, mantendo a maior variância.\n    corr = x[eligible].corr().abs()\n    drop: set[str] = set()\n    for i, left in enumerate(eligible):\n        if left in drop:\n            continue\n        for right in eligible[i + 1:]:\n            if right in drop:\n                continue\n            value = corr.loc[left, right]\n            if np.isfinite(value) and value > corr_threshold:\n                drop.add(left if variance[left] < variance[right] else right)\n    filtered = [c for c in eligible if c not in drop]\n\n    y_h22 = target_matrix(panel, train_origins)[:, -1]\n    raw = x[filtered]\n    medians = raw.median().fillna(0.0)\n    matrix = raw.fillna(medians).to_numpy(float)\n    try:\n        from sklearn.feature_selection import f_regression\n\n        score, pvalue = f_regression(matrix, y_h22)\n    except ImportError:\n        score = np.zeros(len(filtered))\n        pvalue = np.full(len(filtered), np.nan)\n    abs_corr = []\n    for j in range(matrix.shape[1]):\n        value = np.corrcoef(matrix[:, j], y_h22)[0, 1] if np.std(matrix[:, j]) > 0 else 0.0\n        abs_corr.append(abs(value) if np.isfinite(value) else 0.0)\n\n    ranking = pd.DataFrame({\n        "feature": filtered,\n        "f_score_h22": score,\n        "pvalue_h22": pvalue,\n        "abs_corr_h22": abs_corr,\n        "missing_rate_train": [missing[c] for c in filtered],\n        "variance_train": [variance[c] for c in filtered],\n    }).sort_values(["f_score_h22", "abs_corr_h22"], ascending=False).reset_index(drop=True)\n    ranking["rank"] = np.arange(1, len(ranking) + 1)\n    ranking.attrs["counts"] = {\n        "raw_numeric": len(candidates),\n        "after_missing_variance": len(eligible),\n        "after_collinearity": len(filtered),\n    }\n    return ranking\n\n\ndef select_features(panel: pd.DataFrame, train_origins: np.ndarray,\n                    top_k: int | str) -> tuple[list[str], pd.DataFrame]:\n    ranking = rank_features(panel, train_origins)\n    if top_k == "all":\n        selected_exog = ranking.feature.tolist()\n    else:\n        selected_exog = ranking.head(min(int(top_k), len(ranking))).feature.tolist()\n    ranking["selected"] = ranking.feature.isin(selected_exog)\n    # A própria RV histórica é sempre o primeiro canal autorregressivo.\n    return [C.TARGET, *selected_exog], ranking\n\n', 'experiments/putwriting/followup/neural_src/12_tsmixerx_optuna.py': '"""Busca Optuna retomável do TSMixerX diário H=22 na primeira janela 8/2."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sqlite3\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nimport config as C\nfrom feature_selection import select_features\nfrom models import DirectVolatilityRegressor\nfrom utils import (build_sequences, ensure_directories, rolling_821_windows,\n                   save_json, write_split_manifest)\n\n\n\ndef _study_fingerprint(max_steps: int) -> dict:\n    """Identify the data and source used by resumable purged tuning."""\n    src = Path(__file__).resolve().parent\n    files = [Path(__file__).resolve(), src / "models.py", src / "utils.py",\n             src / "feature_selection.py", src / "config.py", C.DAILY_PANEL]\n    return {\n        "sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in files},\n        "split_policy": "purged_train_validation_and_validation_test_h22_v1",\n        "horizon": C.HORIZON, "seed": C.SEED, "max_steps": max_steps,\n        "input_sizes": C.INPUT_SIZES,\n    }\n\ndef _trial_params(trial, max_steps: int) -> dict:\n    return {\n        "model_type": "tsmixerx",\n        "input_size": trial.suggest_categorical("input_size", C.INPUT_SIZES),\n        "n_block": trial.suggest_categorical("n_block", [2, 3, 4, 6]),\n        "ff_dim": trial.suggest_categorical("ff_dim", [32, 64, 128, 256]),\n        "dropout": trial.suggest_categorical("dropout", [0.0, 0.1, 0.2, 0.3]),\n        "learning_rate": trial.suggest_categorical("learning_rate", [1e-4, 3e-4, 1e-3, 3e-3]),\n        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),\n        "loss": trial.suggest_categorical("loss", ["mae", "mse", "huber"]),\n        "max_steps": max_steps,\n        "val_check_steps": 20,\n        "early_stop_patience": 10,\n        "seed": C.SEED,\n    }\n\n\ndef run(target_trials: int = 25, max_steps: int = 400,\n        timeout_minutes: float = 35.0, storage_path: Path | None = None,\n        smoke: bool = False, backup_dir: Path | None = None) -> dict | None:\n    import optuna\n\n    ensure_directories()\n    panel = pd.read_parquet(C.DAILY_PANEL).sort_values("date").reset_index(drop=True)\n    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n    from panel import followup_windows\n    windows = followup_windows(panel)\n    write_split_manifest(panel, windows, C.RESULTS_DIR / "purged_split_manifest.csv")\n    first = windows[0]\n    train, val = first["train_idx"], first["val_idx"]\n    if smoke:\n        target_trials, max_steps = min(target_trials, 3), min(max_steps, 30)\n    storage_path = Path(storage_path or (C.HP_DIR / "tsmixerx_daily_purged.db")).resolve()\n    storage_path.parent.mkdir(parents=True, exist_ok=True)\n    backup_path = None\n    if backup_dir is not None:\n        backup_dir = Path(backup_dir).resolve()\n        backup_dir.mkdir(parents=True, exist_ok=True)\n        backup_path = backup_dir / storage_path.name\n        if backup_path.exists() and not storage_path.exists():\n            with sqlite3.connect(backup_path) as source, sqlite3.connect(storage_path) as destination:\n                source.backup(destination)\n    study_name = f"daily_h22_purged_steps_{max_steps}"\n    study = optuna.create_study(\n        direction="minimize",\n        study_name=study_name,\n        storage=f"sqlite:///{storage_path}",\n        load_if_exists=True,\n        sampler=optuna.samplers.TPESampler(seed=C.SEED),\n        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=100,\n                                           interval_steps=20),\n    )\n    fingerprint = _study_fingerprint(max_steps)\n    previous = study.user_attrs.get("purged_fingerprint")\n    if previous is None:\n        if study.trials:\n            raise RuntimeError("Estudo existente sem fingerprint: não é seguro retomá-lo.")\n        study.set_user_attr("purged_fingerprint", fingerprint)\n    elif previous != fingerprint:\n        raise RuntimeError("Fingerprint do estudo divergente: use novo armazenamento.")\n    history_dir = C.HP_DIR / "trial_histories" / study_name\n    history_dir.mkdir(parents=True, exist_ok=True)\n\n    def objective(trial):\n        top_k = trial.suggest_categorical("n_features", [20, 30, 40, 50, "all"])\n        features, ranking = select_features(panel, train, top_k)\n        params = _trial_params(trial, max_steps)\n        train_origins = train[train >= train[0] + params["input_size"] - 1]\n        train_x, train_y, _ = build_sequences(panel, train_origins, features, params["input_size"])\n        val_x, val_y, _ = build_sequences(panel, val, features, params["input_size"])\n        if not len(train_x) or not len(val_x):\n            raise optuna.TrialPruned("Sem sequências válidas")\n        model = DirectVolatilityRegressor(**params)\n        try:\n            model.fit(train_x, train_y, val_x, val_y, trial=trial)\n        finally:\n            if model.history:\n                model.history_frame(trial_number=trial.number).to_csv(\n                    history_dir / f"trial_{trial.number:04d}.csv", index=False)\n        trial.set_user_attr("selected_features", features)\n        trial.set_user_attr("n_train_sequences", len(train_x))\n        trial.set_user_attr("n_val_sequences", len(val_x))\n        trial.set_user_attr("best_validation_qlike", model.best_validation_qlike)\n        if trial.number == 0:\n            ranking.to_csv(C.HP_DIR / "feature_ranking_initial.csv", index=False)\n            save_json(ranking.attrs.get("counts", {}), C.HP_DIR / "feature_selection_counts.json")\n        return model.best_validation_qlike\n\n    remaining = max(0, target_trials - len(study.trials))\n    def backup_callback(_study, _trial):\n        if backup_path is None:\n            return\n        with sqlite3.connect(storage_path) as source, sqlite3.connect(backup_path) as destination:\n            source.backup(destination)\n\n    if remaining:\n        # Reserva cinco minutos para exportar tabelas, fechar o SQLite e deixar\n        # o notebook executar as células de manifesto/compactação.\n        optimization_seconds = max(1.0, (timeout_minutes - 5.0) * 60.0)\n        study.optimize(objective, n_trials=remaining,\n                       timeout=optimization_seconds,\n                       callbacks=[backup_callback], gc_after_trial=True,\n                       show_progress_bar=True)\n    else:\n        print(f"Meta de {target_trials} trials já alcançada.")\n\n    study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs", "duration")).to_csv(\n        C.HP_DIR / f"trials_{study_name}.csv", index=False)\n    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]\n    if not complete:\n        print("Ainda não há trial completo; execute o notebook novamente.")\n        return None\n    best = study.best_trial\n    top_k = best.params["n_features"]\n    features, ranking = select_features(panel, train, top_k)\n    ranking.to_csv(C.HP_DIR / "feature_ranking_best.csv", index=False)\n    payload = {\n        "study_name": study_name,\n        "study_fingerprint": fingerprint,\n        "storage": str(storage_path),\n        "target_trials": target_trials,\n        "trials_recorded": len(study.trials),\n        "complete_trials": len(complete),\n        "best_trial": best.number,\n        "best_validation_qlike": best.value,\n        "best_params": best.params,\n        "selected_features": features,\n        "max_steps": max_steps,\n        "hp_window": {k: v for k, v in first.items() if not k.endswith("_idx")},\n        "validation_scheme": "fixed first 8-year train / 2-year validation; target boundaries purged by horizon; parameters frozen before all test years",\n        "split_policy": "purged_train_validation_and_validation_test_h22_v1",\n        "pruning": "MedianPruner with trial.report and trial.should_prune at validation checks",\n        "target_transform": "log variance",\n    }\n    save_json(payload, C.HP_DIR / "best_config_purged.json")\n    print(f"Trials: {len(study.trials)}/{target_trials}; completos={len(complete)}")\n    print(f"Melhor trial={best.number}; QLIKE validação={best.value:.6f}")\n    return payload\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--target-trials", type=int, default=25)\n    parser.add_argument("--max-steps", type=int, default=400)\n    parser.add_argument("--timeout-minutes", type=float, default=35.0)\n    parser.add_argument("--storage", type=Path)\n    parser.add_argument("--backup-dir", type=Path,\n                        help="diretório persistente; recebe backup SQLite após cada trial")\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    import torch\n    torch.set_num_threads(2)\n    run(args.target_trials, args.max_steps, args.timeout_minutes, args.storage,\n        args.smoke, args.backup_dir)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'experiments/putwriting/followup/neural_src/13_neural_rolling.py': '"""Ajusta TSMixerX/LSTM nas janelas 8/2/1 com retomada e orçamento de sessão."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\nfrom feature_selection import select_features\nfrom models import DirectVolatilityRegressor\nfrom utils import (build_sequences, ensure_directories, forecasts_long,\n                   rolling_821_windows, save_json, write_split_manifest)\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _guard_resume_policy(results_dir: Path, panel_path: Path, config_path: Path,\n                         split_policy: str, seed: int, max_steps: int) -> dict:\n    """Require one immutable provenance policy for every neural partial.\n\n    Never adopt existing partial files with absent or different provenance, even\n    for a no-resume run: the combiner reads all partials for a model.\n    """\n    results_dir = Path(results_dir)\n    policy_path = results_dir / "neural_resume_policy.json"\n    source_dir = Path(__file__).resolve().parent\n    policy = {\n        "version": 1,\n        "panel_sha256": _sha256(Path(panel_path)),\n        "best_config_sha256": _sha256(Path(config_path)),\n        "split_policy": split_policy,\n        "seed": int(seed),\n        "max_steps": int(max_steps),\n        "source_sha256": {name: _sha256(source_dir / name) for name in\n                          ("13_neural_rolling.py", "models.py", "utils.py",\n                           "feature_selection.py", "config.py")},\n    }\n    partial_dir = results_dir / "forecasts" / "partial"\n    partials = list(partial_dir.glob("tsmixerx_window_*.parquet")) + list(partial_dir.glob("lstm_window_*.parquet"))\n    existing = None\n    if policy_path.exists():\n        try:\n            existing = json.loads(policy_path.read_text(encoding="utf-8"))\n        except (ValueError, OSError) as exc:\n            raise ValueError("Invalid neural resume policy; use a fresh results directory") from exc\n    if partials and existing != policy:\n        raise ValueError("Neural partials have absent or incompatible provenance; use a fresh results directory")\n    if not partials and existing != policy:\n        results_dir.mkdir(parents=True, exist_ok=True)\n        tmp = policy_path.with_suffix(".json.tmp")\n        tmp.write_text(json.dumps(policy, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n        tmp.replace(policy_path)\n    return policy\n\n\ndef _artifact_name(path: Path) -> str:\n    return str(path.relative_to(C.REPO_ROOT)) if path.is_relative_to(C.REPO_ROOT) else str(path)\n\n\ndef _partial_path(model: str, window_id: int) -> Path:\n    path = C.FORECAST_DIR / "partial" / f"{model}_window_{window_id:02d}.parquet"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef _model_params(model_name: str, best: dict, max_steps: int, seed: int) -> dict:\n    params = dict(best["best_params"])\n    params.pop("n_features", None)\n    if model_name == "lstm":\n        return {\n            "model_type": "lstm",\n            "input_size": int(params.get("input_size", 66)),\n            "hidden_size": 64,\n            "dropout": float(params.get("dropout", 0.1)),\n            "learning_rate": 1e-3,\n            "batch_size": int(params.get("batch_size", 64)),\n            "max_steps": max_steps,\n            "loss": "huber",\n            "val_check_steps": 20,\n            "early_stop_patience": 10,\n            "seed": seed,\n        }\n    return {\n        "model_type": "tsmixerx",\n        "input_size": int(params["input_size"]),\n        "n_block": int(params["n_block"]),\n        "ff_dim": int(params["ff_dim"]),\n        "dropout": float(params["dropout"]),\n        "learning_rate": float(params["learning_rate"]),\n        "batch_size": int(params["batch_size"]),\n        "max_steps": max_steps,\n        "loss": params["loss"],\n        "val_check_steps": 20,\n        "early_stop_patience": 10,\n        "seed": seed,\n    }\n\n\ndef _save_inputs(panel: pd.DataFrame, selected: list[str], origins: np.ndarray,\n                 input_size: int, model_name: str, window_id: int) -> Path:\n    start = max(0, int(origins[0]) - input_size + 1)\n    end = int(origins[-1]) + 1\n    columns = ["date", *selected]\n    inputs = panel.loc[start:end - 1, columns].copy()\n    inputs["window_id"] = window_id\n    inputs["model"] = model_name\n    path = C.INPUTS_DIR / f"{model_name}_window_{window_id:02d}.parquet"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    inputs.to_parquet(path, index=False)\n    return path\n\n\ndef _combine(model_name: str) -> None:\n    parts = sorted((_partial_path(model_name, 0).parent).glob(f"{model_name}_window_*.parquet"))\n    if not parts:\n        return\n    out = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)\n    out.to_parquet(C.FORECAST_DIR / f"{model_name}.parquet", index=False)\n    out.to_csv(C.FORECAST_DIR / f"{model_name}.csv", index=False)\n\n\ndef run(models: list[str], max_steps: int | None = None,\n        max_windows: int | None = None, window_ids: list[int] | None = None,\n        time_budget_minutes: float = 20.0, resume: bool = True,\n        seed: int = C.SEED) -> None:\n    ensure_directories()\n    best_path = C.HP_DIR / "best_config_purged.json"\n    if not best_path.exists():\n        raise FileNotFoundError("Execute primeiro 12_tsmixerx_optuna.py; best_config_purged.json ausente.")\n    best = json.loads(best_path.read_text(encoding="utf-8"))\n    if best.get("split_policy") != "purged_train_validation_and_validation_test_h22_v1":\n        raise ValueError("Configuração incompatível com splits purgados; execute nova busca.")\n    max_steps = int(max_steps or best.get("max_steps", 400))\n    _guard_resume_policy(C.RESULTS_DIR, C.DAILY_PANEL, best_path, best["split_policy"], seed, max_steps)\n    panel = pd.read_parquet(C.DAILY_PANEL).sort_values("date").reset_index(drop=True)\n    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n    from panel import followup_windows\n    windows = followup_windows(panel)\n    write_split_manifest(panel, windows, C.RESULTS_DIR / "purged_split_manifest.csv")\n    if window_ids is not None:\n        windows = [w for w in windows if w["window_id"] in window_ids]\n    if max_windows is not None:\n        windows = windows[:max_windows]\n    top_k = best["best_params"].get("n_features", "all")\n    started = time.monotonic()\n    # Não inicia um novo ajuste nos dez minutos finais do orçamento. A margem\n    # é destinada à finalização do ajuste corrente, manifestos e ZIP.\n    start_buffer = min(10.0, max(1.0, time_budget_minutes * 0.20))\n    safe_start_limit = time_budget_minutes - start_buffer\n    manifest_path = C.MODELS_DIR / "neural_manifest.json"\n    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {"artifacts": []}\n\n    for w in windows:\n        for model_name in models:\n            model_name = model_name.lower()\n            partial = _partial_path(model_name, w["window_id"])\n            if resume and partial.exists():\n                print(f"Skip {model_name} janela {w[\'window_id\']:02d}: já concluída")\n                continue\n            elapsed = (time.monotonic() - started) / 60.0\n            if elapsed >= safe_start_limit:\n                print("Limite seguro atingido. Nenhum novo ajuste será iniciado; "\n                      "os artefatos concluídos já estão no armazenamento da sessão.")\n                for name in models:\n                    _combine(name.lower())\n                return\n\n            params = _model_params(model_name, best, max_steps, seed)\n            input_size = params["input_size"]\n            selected, ranking = select_features(panel, w["train_idx"], top_k)\n            ranking_dir = C.RESULTS_DIR / "feature_selection"\n            ranking_dir.mkdir(parents=True, exist_ok=True)\n            ranking.to_csv(ranking_dir / f"ranking_{model_name}_window_{w[\'window_id\']:02d}.csv", index=False)\n\n            # O contexto de uma sequência de treino não pode começar antes da\n            # janela de oito anos. Validação e teste podem usar o histórico já\n            # observado das partições anteriores.\n            train_origins = w["train_idx"][w["train_idx"] >= w["train_idx"][0] + input_size - 1]\n            train_x, train_y, _ = build_sequences(panel, train_origins, selected, input_size)\n            val_x, val_y, _ = build_sequences(panel, w["val_idx"], selected, input_size)\n            test_origins = np.asarray([i for i in w["test_idx"] if i >= input_size - 1], dtype=int)\n            raw = panel[selected].apply(pd.to_numeric, errors="coerce").to_numpy(float)\n            test_x = np.asarray([raw[i-input_size+1:i+1] for i in test_origins], dtype=np.float32)\n            print(f"{model_name} janela {w[\'window_id\']:02d} ({w[\'test_year\']}): "\n                  f"train={len(train_x)}, val={len(val_x)}, test={len(test_x)}, features={len(selected)}")\n\n            estimator = DirectVolatilityRegressor(**params)\n            estimator.fit(train_x, train_y, val_x, val_y)\n            pred_raw = estimator.predict(test_x)\n            train_pred = estimator.predict(train_x)\n            factors = np.mean(train_y / np.maximum(train_pred, C.EPS), axis=0)\n            pred = pred_raw * factors\n            artifact = C.MODELS_DIR / model_name / f"{model_name}_window_{w[\'window_id\']:02d}.pt"\n            metadata = {\n                **{k: v for k, v in w.items() if not k.endswith("_idx")},\n                "selected_features": selected,\n                "target": C.TARGET,\n                "horizon": C.HORIZON,\n                "target_transform": "log variance",\n                "smearing_factors_by_horizon": factors.tolist(),\n                "smearing_estimation": "mean exp(log observed training target minus fitted log prediction), in-sample, one factor per horizon; same fitted model",\n                "split_policy": best["split_policy"],\n            }\n            estimator.save(artifact, selected, metadata)\n            history = estimator.history_frame(model=model_name, window_id=w["window_id"],\n                                                test_year=w["test_year"], seed=seed)\n            history.to_csv(C.HISTORY_DIR / f"{model_name}_window_{w[\'window_id\']:02d}.csv", index=False)\n            inputs_path = _save_inputs(panel, selected, test_origins, input_size,\n                                       model_name, w["window_id"])\n\n            # Valida imediatamente que o artefato sobrevive à serialização.\n            loaded, _ = DirectVolatilityRegressor.load(artifact)\n            reload_pred = loaded.predict(test_x[:min(2, len(test_x))])\n            if not np.allclose(reload_pred, pred_raw[:len(reload_pred)], rtol=1e-5, atol=1e-8):\n                raise RuntimeError(f"Reload divergente em {artifact}")\n\n            extra_columns = [c for c in selected if c != C.TARGET]\n            for common in ["vix", "spx_close", "close_variance_forward_22",\n                           "close_volatility_forward_22"]:\n                if common not in extra_columns:\n                    extra_columns.append(common)\n            rows = []\n            for i, origin in enumerate(test_origins):\n                for h in range(1, C.HORIZON + 1):\n                    j = int(origin) + h\n                    y = float(panel.loc[j, C.TARGET]) if j < len(panel) and pd.notna(panel.loc[j, C.TARGET]) else np.nan\n                    rows.append({"model": "TSMixerX" if model_name == "tsmixerx" else "LSTM", "window_id": w["window_id"], "origin_date": panel.loc[origin, "date"], "target_date": panel.loc[j, "date"] if j < len(panel) else pd.NaT, "horizon": h, "y": y, "y_hat": float(pred[i,h-1]), "y_hat_raw": float(pred_raw[i,h-1]), "y_hat_log": float(np.log(pred_raw[i,h-1])), "smearing_factor": float(factors[h-1]), "artifact_path": _artifact_name(artifact), **{f"x_{c}": panel.loc[origin,c] for c in extra_columns if c in panel}})\n            frame = pd.DataFrame(rows)\n            frame["error"] = frame.y_hat - frame.y\n            frame["squared_error"] = frame.error ** 2\n            ratio = frame.y.clip(lower=C.EPS) / frame.y_hat.clip(lower=C.EPS)\n            frame["qlike_loss"] = ratio - np.log(ratio) - 1\n            frame.to_parquet(partial, index=False)\n            frame.to_csv(partial.with_suffix(".csv"), index=False)\n            manifest["artifacts"].append({\n                "model": model_name,\n                "window_id": w["window_id"],\n                "test_year": w["test_year"],\n                "artifact": _artifact_name(artifact),\n                "inputs": _artifact_name(inputs_path),\n                "history": _artifact_name(C.HISTORY_DIR / f"{model_name}_window_{w[\'window_id\']:02d}.csv"),\n                "best_step": int(history.loc[history.validation_qlike.idxmin(), "step"]),\n                "stopped_early": bool(history.stopped_early.any()),\n                "reload_verified": True,\n            })\n            save_json(manifest, manifest_path)\n            _combine(model_name)\n    for model_name in models:\n        _combine(model_name.lower())\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--models", nargs="+", choices=["tsmixerx", "lstm"],\n                        default=["tsmixerx", "lstm"])\n    parser.add_argument("--max-steps", type=int)\n    parser.add_argument("--max-windows", type=int)\n    parser.add_argument("--window-ids", type=int, nargs="*")\n    parser.add_argument("--time-budget-minutes", type=float, default=20.0)\n    parser.add_argument("--seed", type=int, default=C.SEED)\n    parser.add_argument("--no-resume", action="store_true")\n    args = parser.parse_args()\n    import torch\n    torch.set_num_threads(2)\n    run(args.models, args.max_steps, args.max_windows, args.window_ids,\n        args.time_budget_minutes, not args.no_resume, args.seed)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'experiments/putwriting/followup/neural_src/utils.py': '"""Alvos, splits, matrizes e métricas do experimento diário H=22."""\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\n\n\ndef ensure_directories() -> None:\n    for path in [C.DATA_DIR, C.RESULTS_DIR, C.FORECAST_DIR, C.INPUTS_DIR,\n                 C.METRICS_DIR, C.FIGURES_DIR, C.MODELS_DIR, C.HP_DIR,\n                 C.HISTORY_DIR, C.EXPLAIN_DIR]:\n        path.mkdir(parents=True, exist_ok=True)\n\n\ndef set_seed(seed: int = C.SEED) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef compute_yang_zhang(ohlc: pd.DataFrame, window: int = C.RV_WINDOW,\n                       annualization: int = C.ANNUALIZATION) -> pd.Series:\n    """Yang-Zhang padrão em janela móvel, retornando variância anualizada.\n\n    A implementação combina variância overnight, variância open-to-close e o\n    estimador Rogers-Satchell. O peso é\n    ``k=0.34/(1.34+(n+1)/(n-1))``.\n    """\n    required = ["spx_open", "spx_high", "spx_low", "spx_close"]\n    missing = [c for c in required if c not in ohlc]\n    if missing:\n        raise KeyError(f"OHLC ausente: {missing}")\n    if window < 2:\n        raise ValueError("Yang-Zhang requer window >= 2.")\n\n    p = ohlc[required].apply(pd.to_numeric, errors="coerce").astype(float)\n    valid = (p > 0).all(axis=1)\n    p = p.where(valid)\n\n    overnight = np.log(p["spx_open"] / p["spx_close"].shift(1))\n    open_close = np.log(p["spx_close"] / p["spx_open"])\n    high_open = np.log(p["spx_high"] / p["spx_open"])\n    low_open = np.log(p["spx_low"] / p["spx_open"])\n    rs_daily = high_open * (high_open - open_close) + low_open * (low_open - open_close)\n\n    var_overnight = overnight.rolling(window, min_periods=window).var(ddof=1)\n    var_open_close = open_close.rolling(window, min_periods=window).var(ddof=1)\n    var_rs = rs_daily.rolling(window, min_periods=window).mean()\n    k = 0.34 / (1.34 + (window + 1.0) / (window - 1.0))\n    yz = var_overnight + k * var_open_close + (1.0 - k) * var_rs\n    yz = (yz * annualization).where(np.isfinite(yz) & (yz > 0))\n    return yz.rename(C.TARGET)\n\n\ndef add_forward_close_variance(frame: pd.DataFrame, horizon: int = C.HORIZON) -> pd.DataFrame:\n    """Adiciona diagnóstico aditivo de variância close-to-close futura."""\n    out = frame.copy()\n    r = np.log(pd.to_numeric(out["spx_close"], errors="coerce") /\n               pd.to_numeric(out["spx_close"], errors="coerce").shift(1))\n    future_sum = sum(r.shift(-h).pow(2) for h in range(1, horizon + 1))\n    variance = future_sum * C.ANNUALIZATION / horizon\n    out["close_variance_forward_22"] = variance\n    out["close_volatility_forward_22"] = np.sqrt(variance.clip(lower=0))\n    return out\n\n\ndef valid_origin_mask(panel: pd.DataFrame, horizon: int = C.HORIZON) -> np.ndarray:\n    y = panel[C.TARGET].to_numpy(float)\n    valid = np.isfinite(y)\n    mask = np.zeros(len(panel), dtype=bool)\n    for pos in range(len(panel) - horizon):\n        mask[pos] = valid[pos] and valid[pos + 1:pos + horizon + 1].all()\n    return mask\n\n\ndef target_matrix(panel: pd.DataFrame, origins: np.ndarray,\n                  horizon: int = C.HORIZON) -> np.ndarray:\n    y = panel[C.TARGET].to_numpy(float)\n    return np.asarray([y[int(pos) + 1:int(pos) + horizon + 1] for pos in origins], dtype=float)\n\n\ndef rolling_821_windows(panel: pd.DataFrame, purge_boundaries: bool = False) -> list[dict]:\n    """Janelas anuais com 8 anos de treino, 2 de validação e 1 de teste."""\n    dates = pd.to_datetime(panel["date"])\n    years = dates.dt.year.to_numpy()\n    valid_origin = valid_origin_mask(panel)\n    first_year, last_year = int(years.min()), int(years.max())\n    first_test = first_year + C.TRAIN_YEARS + C.VAL_YEARS\n    windows: list[dict] = []\n    for window_id, test_year in enumerate(range(first_test, last_year + 1, C.STEP_YEARS)):\n        train_start = test_year - C.VAL_YEARS - C.TRAIN_YEARS\n        train_end = test_year - C.VAL_YEARS\n        val_end = test_year\n        train = np.flatnonzero((years >= train_start) & (years < train_end) & valid_origin)\n        val = np.flatnonzero((years >= train_end) & (years < val_end) & valid_origin)\n        test = np.flatnonzero((years == test_year) & valid_origin)\n        if purge_boundaries:\n            # Every target for an origin must be known before the next split.\n            val_boundary = int(np.flatnonzero(years >= train_end)[0])\n            test_boundary = int(np.flatnonzero(years >= val_end)[0])\n            train = train[train + C.HORIZON < val_boundary]\n            val = val[val + C.HORIZON < test_boundary]\n        if len(train) and len(val) and len(test):\n            windows.append({\n                "window_id": window_id,\n                "test_year": test_year,\n                "train_year_start": train_start,\n                "train_year_end": train_end - 1,\n                "val_year_start": train_end,\n                "val_year_end": val_end - 1,\n                "train_idx": train,\n                "val_idx": val,\n                "test_idx": test,\n            })\n    return windows\n\n\ndef write_split_manifest(panel: pd.DataFrame, windows: list[dict],\n                         path: Path = C.SPLIT_MANIFEST) -> pd.DataFrame:\n    rows = []\n    for w in windows:\n        row = {k: v for k, v in w.items() if not k.endswith("_idx")}\n        for role in ["train", "val", "test"]:\n            idx = w[f"{role}_idx"]\n            row[f"{role}_start"] = panel.loc[idx[0], "date"]\n            row[f"{role}_end"] = panel.loc[idx[-1], "date"]\n            row[f"{role}_origins"] = len(idx)\n        rows.append(row)\n    manifest = pd.DataFrame(rows)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(path, index=False)\n    return manifest\n\n\ndef feature_columns(panel: pd.DataFrame) -> list[str]:\n    return [c for c in panel.columns\n            if c not in C.NON_FEATURE_COLUMNS\n            and c not in C.EXCLUDED_MAIN\n            and pd.api.types.is_numeric_dtype(panel[c])]\n\n\ndef build_sequences(panel: pd.DataFrame, origins: np.ndarray, features: list[str],\n                    input_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Retorna X, Y e origens válidas sem cruzar o início da base."""\n    raw = panel[features].apply(pd.to_numeric, errors="coerce").to_numpy(float)\n    y = panel[C.TARGET].to_numpy(float)\n    seq, targets, kept = [], [], []\n    for origin in origins:\n        origin = int(origin)\n        lo = origin - input_size + 1\n        if lo < 0 or origin + C.HORIZON >= len(panel):\n            continue\n        target = y[origin + 1:origin + C.HORIZON + 1]\n        if np.isfinite(target).all():\n            seq.append(raw[lo:origin + 1])\n            targets.append(target)\n            kept.append(origin)\n    return np.asarray(seq, dtype=np.float32), np.asarray(targets, dtype=np.float32), np.asarray(kept, dtype=int)\n\n\ndef forecasts_long(panel: pd.DataFrame, origins: np.ndarray, predictions: np.ndarray,\n                   model: str, window_id: int, artifact_path: str | None = None,\n                   extra_origin_columns: list[str] | None = None) -> pd.DataFrame:\n    rows = []\n    extras = extra_origin_columns or []\n    for i, origin in enumerate(origins):\n        origin = int(origin)\n        for h in range(1, C.HORIZON + 1):\n            target_pos = origin + h\n            row = {\n                "model": model,\n                "window_id": int(window_id),\n                "origin_date": panel.loc[origin, "date"],\n                "target_date": panel.loc[target_pos, "date"],\n                "horizon": h,\n                "y": float(panel.loc[target_pos, C.TARGET]),\n                "y_hat": float(max(predictions[i, h - 1], C.EPS)),\n                "artifact_path": artifact_path,\n            }\n            for col in extras:\n                if col in panel:\n                    row[f"x_{col}"] = panel.loc[origin, col]\n            rows.append(row)\n    out = pd.DataFrame(rows)\n    out["error"] = out["y_hat"] - out["y"]\n    out["squared_error"] = out["error"].pow(2)\n    ratio = out["y"].clip(lower=C.EPS) / out["y_hat"].clip(lower=C.EPS)\n    out["qlike_loss"] = ratio - np.log(ratio) - 1.0\n    return out\n\n\ndef save_json(payload: dict, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")\n\n\ndef qlike_by_horizon(y: np.ndarray, pred: np.ndarray) -> np.ndarray:\n    ratio = np.maximum(y, C.EPS) / np.maximum(pred, C.EPS)\n    return np.mean(ratio - np.log(ratio) - 1.0, axis=0)\n\n\ndef aggregate_qlike(y: np.ndarray, pred: np.ndarray) -> float:\n    return float(np.mean(qlike_by_horizon(y, pred)))\n\n', 'experiments/putwriting/followup/panel.py': '"""Rebuild the 8/2/1 forecast panel after auditing the candidate features."""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom importlib import import_module\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nDAILY_SRC = ROOT / "experiments/daily_h22/src"\nif str(DAILY_SRC) not in sys.path:\n    sys.path.insert(0, str(DAILY_SRC))\n\nimport config as C\nfrom utils import rolling_821_windows, write_split_manifest\n\nRESULTS = HERE / "results"\nEXCLUDED = ("ret_nky_simple", "ret_ukx_simple", "ret_wti_simple")\nEXCLUDED_FEATURES = (*EXCLUDED, "vix_term_spread")\n\n\ndef followup_windows(panel: pd.DataFrame) -> list[dict]:\n    """Purge train/validation targets and generate test predictions through the last close."""\n    dates = pd.to_datetime(panel["date"])\n    y = panel[C.TARGET].to_numpy(float)\n    windows = rolling_821_windows(panel, purge_boundaries=True)\n    for window in windows:\n        window["test_idx"] = np.flatnonzero(\n            (dates.dt.year.to_numpy() == window["test_year"]) & np.isfinite(y) & (y > 0)\n        )\n    if not windows or windows[-1]["test_year"] != int(dates.dt.year.max()):\n        raise ValueError("Last calendar year has no admissible 8/2/1 fold")\n    return windows\n\n\ndef prepare(workbook: Path, output: Path) -> pd.DataFrame:\n    prep = import_module("10_data_prep_daily")\n    daily = pd.read_excel(workbook, sheet_name="painel_diario")\n    macro = pd.read_excel(workbook, sheet_name="macro")\n    suspect = pd.concat(\n        [\n            daily.loc[daily[column].le(-1), ["date", column]]\n            .rename(columns={column: "value"})\n            .assign(feature=column)\n            for column in EXCLUDED\n        ],\n        ignore_index=True,\n    )\n    # The term spread exactly reconstructs the excluded VIX3M from the VIX.\n    clean = prep._clean_daily(daily.drop(columns=list(EXCLUDED_FEATURES)))\n    panel = prep._merge_macro_conservative(clean, macro)\n    panel[C.TARGET] = prep.compute_yang_zhang(panel)\n    panel = prep.add_forward_close_variance(panel)\n    panel.insert(1, "unique_id", "sp500")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    panel.to_parquet(output, index=False)\n    split = write_split_manifest(panel, followup_windows(panel), output.parent / "split_manifest.csv")\n    suspect.groupby("feature").agg(\n        values_le_minus_one=("value", "size"),\n        values_equal_minus_one=("value", lambda x: x.eq(-1).sum()),\n        minimum=("value", "min"),\n    ).to_csv(output.parent / "invalid_simple_returns.csv")\n    print(f"{len(panel)} sessions, {len(split)} folds, through {panel.date.max().date()}", flush=True)\n    return panel\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--workbook", type=Path, default=C.SOURCE_XLSX)\n    parser.add_argument("--output", type=Path, default=RESULTS / "panel.parquet")\n    args = parser.parse_args()\n    prepare(args.workbook, args.output)\n', 'experiments/daily_h22/src/config.py': '"""Configuração isolada do experimento diário H=22."""\n\nimport os\nfrom pathlib import Path\n\nEXPERIMENT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = Path(__file__).resolve().parents[3]\n\nSOURCE_XLSX = REPO_ROOT / "base de dados dissertação.xlsx"\nDATA_DIR = Path(os.environ.get("DAILY_H22_DATA_DIR", EXPERIMENT_ROOT / "data"))\nDAILY_PANEL = DATA_DIR / "daily_panel.parquet"\nSPLIT_MANIFEST = DATA_DIR / "split_manifest.csv"\n\nRESULTS_DIR = Path(os.environ.get("DAILY_H22_RESULTS_DIR", EXPERIMENT_ROOT / "results"))\nFORECAST_DIR = RESULTS_DIR / "forecasts"\nINPUTS_DIR = RESULTS_DIR / "inputs"\nMETRICS_DIR = RESULTS_DIR / "metrics"\nFIGURES_DIR = RESULTS_DIR / "figures"\nMODELS_DIR = RESULTS_DIR / "models"\nHP_DIR = RESULTS_DIR / "hp_search"\nHISTORY_DIR = RESULTS_DIR / "training_history"\nEXPLAIN_DIR = RESULTS_DIR / "explainability"\n\nTARGET = "rv_yz_22"\nHORIZON = 22\nRV_WINDOW = 22\nANNUALIZATION = 252\n\nTRAIN_YEARS = 8\nVAL_YEARS = 2\nTEST_YEARS = 1\nSTEP_YEARS = 1\n\n# Conversão dos lags mensais acordados {1,3,6,12,24} para pregões.\nINPUT_SIZES = [22, 66, 132, 264, 528]\n\nSEED = 42\nEPS = 1e-10\n\nPRICE_RETURN_PAIRS = {\n    "spx_close": "ret_spx_log",\n    "ndx_close": "ret_ndx_log",\n    "indu_close": "ret_indu_log",\n    "nky_close": "ret_nky_log",\n    "ukx_close": "ret_ukx_log",\n    "wti_close": "ret_wti_log",\n    "gold_close": "ret_gold_log",\n    "dxy": "ret_dxy_log",\n}\n\nMONTHLY_MACRO = ["industrial_production", "cpi", "unemployment"]\nWEEKLY_MACRO = ["stress_index"]\n\n# VIX3M é deliberadamente excluído da análise principal 2000-2025.\nEXCLUDED_MAIN = {\n    "vix3m",\n    "retorno_acum_22d",\n    "rv_semanal",\n    "rv_mensal",\n    "rolling_std_22d",\n}\n\nNON_FEATURE_COLUMNS = {\n    "date",\n    "unique_id",\n    TARGET,\n    "close_variance_forward_22",\n    "close_volatility_forward_22",\n}\n', 'experiments/daily_h22/src/utils.py': '"""Alvos, splits, matrizes e métricas do experimento diário H=22."""\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\n\n\ndef ensure_directories() -> None:\n    for path in [C.DATA_DIR, C.RESULTS_DIR, C.FORECAST_DIR, C.INPUTS_DIR,\n                 C.METRICS_DIR, C.FIGURES_DIR, C.MODELS_DIR, C.HP_DIR,\n                 C.HISTORY_DIR, C.EXPLAIN_DIR]:\n        path.mkdir(parents=True, exist_ok=True)\n\n\ndef set_seed(seed: int = C.SEED) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef compute_yang_zhang(ohlc: pd.DataFrame, window: int = C.RV_WINDOW,\n                       annualization: int = C.ANNUALIZATION) -> pd.Series:\n    """Yang-Zhang padrão em janela móvel, retornando variância anualizada.\n\n    A implementação combina variância overnight, variância open-to-close e o\n    estimador Rogers-Satchell. O peso é\n    ``k=0.34/(1.34+(n+1)/(n-1))``.\n    """\n    required = ["spx_open", "spx_high", "spx_low", "spx_close"]\n    missing = [c for c in required if c not in ohlc]\n    if missing:\n        raise KeyError(f"OHLC ausente: {missing}")\n    if window < 2:\n        raise ValueError("Yang-Zhang requer window >= 2.")\n\n    p = ohlc[required].apply(pd.to_numeric, errors="coerce").astype(float)\n    valid = (p > 0).all(axis=1)\n    p = p.where(valid)\n\n    overnight = np.log(p["spx_open"] / p["spx_close"].shift(1))\n    open_close = np.log(p["spx_close"] / p["spx_open"])\n    high_open = np.log(p["spx_high"] / p["spx_open"])\n    low_open = np.log(p["spx_low"] / p["spx_open"])\n    rs_daily = high_open * (high_open - open_close) + low_open * (low_open - open_close)\n\n    var_overnight = overnight.rolling(window, min_periods=window).var(ddof=1)\n    var_open_close = open_close.rolling(window, min_periods=window).var(ddof=1)\n    var_rs = rs_daily.rolling(window, min_periods=window).mean()\n    k = 0.34 / (1.34 + (window + 1.0) / (window - 1.0))\n    yz = var_overnight + k * var_open_close + (1.0 - k) * var_rs\n    yz = (yz * annualization).where(np.isfinite(yz) & (yz > 0))\n    return yz.rename(C.TARGET)\n\n\ndef add_forward_close_variance(frame: pd.DataFrame, horizon: int = C.HORIZON) -> pd.DataFrame:\n    """Adiciona diagnóstico aditivo de variância close-to-close futura."""\n    out = frame.copy()\n    r = np.log(pd.to_numeric(out["spx_close"], errors="coerce") /\n               pd.to_numeric(out["spx_close"], errors="coerce").shift(1))\n    future_sum = sum(r.shift(-h).pow(2) for h in range(1, horizon + 1))\n    variance = future_sum * C.ANNUALIZATION / horizon\n    out["close_variance_forward_22"] = variance\n    out["close_volatility_forward_22"] = np.sqrt(variance.clip(lower=0))\n    return out\n\n\ndef valid_origin_mask(panel: pd.DataFrame, horizon: int = C.HORIZON) -> np.ndarray:\n    y = panel[C.TARGET].to_numpy(float)\n    valid = np.isfinite(y)\n    mask = np.zeros(len(panel), dtype=bool)\n    for pos in range(len(panel) - horizon):\n        mask[pos] = valid[pos] and valid[pos + 1:pos + horizon + 1].all()\n    return mask\n\n\ndef target_matrix(panel: pd.DataFrame, origins: np.ndarray,\n                  horizon: int = C.HORIZON) -> np.ndarray:\n    y = panel[C.TARGET].to_numpy(float)\n    return np.asarray([y[int(pos) + 1:int(pos) + horizon + 1] for pos in origins], dtype=float)\n\n\ndef rolling_821_windows(panel: pd.DataFrame, purge_boundaries: bool = False) -> list[dict]:\n    """Janelas anuais com 8 anos de treino, 2 de validação e 1 de teste."""\n    dates = pd.to_datetime(panel["date"])\n    years = dates.dt.year.to_numpy()\n    valid_origin = valid_origin_mask(panel)\n    first_year, last_year = int(years.min()), int(years.max())\n    first_test = first_year + C.TRAIN_YEARS + C.VAL_YEARS\n    windows: list[dict] = []\n    for window_id, test_year in enumerate(range(first_test, last_year + 1, C.STEP_YEARS)):\n        train_start = test_year - C.VAL_YEARS - C.TRAIN_YEARS\n        train_end = test_year - C.VAL_YEARS\n        val_end = test_year\n        train = np.flatnonzero((years >= train_start) & (years < train_end) & valid_origin)\n        val = np.flatnonzero((years >= train_end) & (years < val_end) & valid_origin)\n        test = np.flatnonzero((years == test_year) & valid_origin)\n        if purge_boundaries:\n            # Every target for an origin must be known before the next split.\n            val_boundary = int(np.flatnonzero(years >= train_end)[0])\n            test_boundary = int(np.flatnonzero(years >= val_end)[0])\n            train = train[train + C.HORIZON < val_boundary]\n            val = val[val + C.HORIZON < test_boundary]\n        if len(train) and len(val) and len(test):\n            windows.append({\n                "window_id": window_id,\n                "test_year": test_year,\n                "train_year_start": train_start,\n                "train_year_end": train_end - 1,\n                "val_year_start": train_end,\n                "val_year_end": val_end - 1,\n                "train_idx": train,\n                "val_idx": val,\n                "test_idx": test,\n            })\n    return windows\n\n\ndef write_split_manifest(panel: pd.DataFrame, windows: list[dict],\n                         path: Path = C.SPLIT_MANIFEST) -> pd.DataFrame:\n    rows = []\n    for w in windows:\n        row = {k: v for k, v in w.items() if not k.endswith("_idx")}\n        for role in ["train", "val", "test"]:\n            idx = w[f"{role}_idx"]\n            row[f"{role}_start"] = panel.loc[idx[0], "date"]\n            row[f"{role}_end"] = panel.loc[idx[-1], "date"]\n            row[f"{role}_origins"] = len(idx)\n        rows.append(row)\n    manifest = pd.DataFrame(rows)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(path, index=False)\n    return manifest\n\n\ndef feature_columns(panel: pd.DataFrame) -> list[str]:\n    return [c for c in panel.columns\n            if c not in C.NON_FEATURE_COLUMNS\n            and c not in C.EXCLUDED_MAIN\n            and pd.api.types.is_numeric_dtype(panel[c])]\n\n\ndef build_sequences(panel: pd.DataFrame, origins: np.ndarray, features: list[str],\n                    input_size: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Retorna X, Y e origens válidas sem cruzar o início da base."""\n    raw = panel[features].apply(pd.to_numeric, errors="coerce").to_numpy(float)\n    y = panel[C.TARGET].to_numpy(float)\n    seq, targets, kept = [], [], []\n    for origin in origins:\n        origin = int(origin)\n        lo = origin - input_size + 1\n        if lo < 0 or origin + C.HORIZON >= len(panel):\n            continue\n        target = y[origin + 1:origin + C.HORIZON + 1]\n        if np.isfinite(target).all():\n            seq.append(raw[lo:origin + 1])\n            targets.append(target)\n            kept.append(origin)\n    return np.asarray(seq, dtype=np.float32), np.asarray(targets, dtype=np.float32), np.asarray(kept, dtype=int)\n\n\ndef forecasts_long(panel: pd.DataFrame, origins: np.ndarray, predictions: np.ndarray,\n                   model: str, window_id: int, artifact_path: str | None = None,\n                   extra_origin_columns: list[str] | None = None) -> pd.DataFrame:\n    rows = []\n    extras = extra_origin_columns or []\n    for i, origin in enumerate(origins):\n        origin = int(origin)\n        for h in range(1, C.HORIZON + 1):\n            target_pos = origin + h\n            row = {\n                "model": model,\n                "window_id": int(window_id),\n                "origin_date": panel.loc[origin, "date"],\n                "target_date": panel.loc[target_pos, "date"],\n                "horizon": h,\n                "y": float(panel.loc[target_pos, C.TARGET]),\n                "y_hat": float(max(predictions[i, h - 1], C.EPS)),\n                "artifact_path": artifact_path,\n            }\n            for col in extras:\n                if col in panel:\n                    row[f"x_{col}"] = panel.loc[origin, col]\n            rows.append(row)\n    out = pd.DataFrame(rows)\n    out["error"] = out["y_hat"] - out["y"]\n    out["squared_error"] = out["error"].pow(2)\n    ratio = out["y"].clip(lower=C.EPS) / out["y_hat"].clip(lower=C.EPS)\n    out["qlike_loss"] = ratio - np.log(ratio) - 1.0\n    return out\n\n\ndef save_json(payload: dict, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")\n\n\ndef qlike_by_horizon(y: np.ndarray, pred: np.ndarray) -> np.ndarray:\n    ratio = np.maximum(y, C.EPS) / np.maximum(pred, C.EPS)\n    return np.mean(ratio - np.log(ratio) - 1.0, axis=0)\n\n\ndef aggregate_qlike(y: np.ndarray, pred: np.ndarray) -> float:\n    return float(np.mean(qlike_by_horizon(y, pred)))\n\n', 'experiments/daily_h22/src/10_data_prep_daily.py': '"""Cria o painel diário H=22 com Yang-Zhang corrigido e splits 8/2/1."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nimport config as C\nfrom utils import (add_forward_close_variance, compute_yang_zhang,\n                   ensure_directories, feature_columns, rolling_821_windows,\n                   write_split_manifest)\n\n\ndef _clean_daily(frame: pd.DataFrame) -> pd.DataFrame:\n    out = frame.copy()\n    out.columns = [str(c).strip() for c in out]\n    out = out.drop(columns=[c for c in out if c.lower().startswith("unnamed")], errors="ignore")\n    out["date"] = pd.to_datetime(out["date"], errors="coerce")\n    out = out.dropna(subset=["date"]).sort_values("date").drop_duplicates("date").reset_index(drop=True)\n\n    for col in ["spx_open", "spx_high", "spx_low", "spx_close"]:\n        out[col] = pd.to_numeric(out[col], errors="coerce")\n    out = out.loc[(out[["spx_open", "spx_high", "spx_low", "spx_close"]] > 0).all(axis=1)].reset_index(drop=True)\n\n    for col in ["nky_close", "ukx_close", "baa", "aaa"]:\n        if col in out:\n            out[col] = pd.to_numeric(out[col], errors="coerce").ffill()\n    for price_col, return_col in C.PRICE_RETURN_PAIRS.items():\n        if price_col in out:\n            price = pd.to_numeric(out[price_col], errors="coerce").where(lambda s: s > 0)\n            out[return_col] = np.log(price / price.shift(1)).replace([np.inf, -np.inf], np.nan)\n\n    # Estas séries serão reconstruídas com uma data de disponibilidade\n    # conservadora, em vez de usar o mês de referência como se fosse release.\n    out = out.drop(columns=[*C.MONTHLY_MACRO, *C.WEEKLY_MACRO], errors="ignore")\n    if "epu" in out:\n        out["epu"] = pd.to_numeric(out["epu"], errors="coerce").shift(22)\n    return out\n\n\ndef _merge_macro_conservative(daily: pd.DataFrame, macro: pd.DataFrame) -> pd.DataFrame:\n    out = daily.sort_values("date").copy()\n    m = macro.copy()\n    m.columns = [str(c).strip() for c in m]\n    if "date_ref_mensal" in m:\n        mm_cols = [c for c in C.MONTHLY_MACRO if c in m]\n        mm = m[["date_ref_mensal", *mm_cols]].dropna(subset=["date_ref_mensal"]).copy()\n        mm["date_ref_mensal"] = pd.to_datetime(mm["date_ref_mensal"])\n        # Sem release dates individuais na planilha, adotamos um mês completo\n        # de atraso. A hipótese fica registrada no manifesto.\n        mm["availability_date"] = mm["date_ref_mensal"] + pd.offsets.MonthEnd(1)\n        mm = mm.sort_values("availability_date").drop_duplicates("availability_date", keep="last")\n        out = pd.merge_asof(out, mm[["availability_date", *mm_cols]],\n                            left_on="date", right_on="availability_date", direction="backward")\n        out = out.drop(columns="availability_date")\n    if "date_ref_semanal" in m and "stress_index" in m:\n        mw = m[["date_ref_semanal", "stress_index"]].dropna(subset=["date_ref_semanal"]).copy()\n        mw["date_ref_semanal"] = pd.to_datetime(mw["date_ref_semanal"])\n        mw["availability_date"] = mw["date_ref_semanal"] + pd.Timedelta(days=7)\n        mw = mw.sort_values("availability_date").drop_duplicates("availability_date", keep="last")\n        out = pd.merge_asof(out.sort_values("date"), mw[["availability_date", "stress_index"]],\n                            left_on="date", right_on="availability_date", direction="backward")\n        out = out.drop(columns="availability_date")\n    return out\n\n\ndef prepare(input_path=C.SOURCE_XLSX, output=C.DAILY_PANEL) -> pd.DataFrame:\n    ensure_directories()\n    daily = pd.read_excel(input_path, sheet_name="painel_diario")\n    macro = pd.read_excel(input_path, sheet_name="macro")\n    daily = _merge_macro_conservative(_clean_daily(daily), macro)\n    daily[C.TARGET] = compute_yang_zhang(daily)\n    daily = add_forward_close_variance(daily)\n    daily.insert(1, "unique_id", "sp500")\n    daily.to_parquet(output, index=False)\n    windows = rolling_821_windows(daily)\n    write_split_manifest(daily, windows)\n    return daily\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input", default=C.SOURCE_XLSX)\n    parser.add_argument("--output", default=C.DAILY_PANEL)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    panel = prepare(args.input, args.output)\n    windows = rolling_821_windows(panel)\n    print(f"Painel salvo em {args.output}; shape={panel.shape}")\n    print(f"Período: {panel.date.min().date()} a {panel.date.max().date()}")\n    print(f"RV_YZ válida: {panel[C.TARGET].notna().sum()}; janelas 8/2/1: {len(windows)}")\n    print(f"Features principais: {len(feature_columns(panel))}; VIX3M excluído: {\'vix3m\' not in feature_columns(panel)}")\n    if args.smoke:\n        assert len(windows) == 16, f"Esperadas 16 janelas, obtidas {len(windows)}"\n        assert (panel[C.TARGET].dropna() > 0).all()\n        first = windows[0]\n        print({k: v for k, v in first.items() if not k.endswith("_idx")})\n        print(panel.loc[panel.date.between("2020-02-01", "2020-05-31"), ["date", C.TARGET]].nlargest(5, C.TARGET).to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n\n'}
for name, content in files.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
print("Reconstructed", len(files), "source files")


In [ ]:
from google.colab import files as colab_files
uploaded = colab_files.upload()
assert len(uploaded) == 1, "Upload exactly one original workbook"
workbook = str((root / next(iter(uploaded))).resolve())
assert workbook.lower().endswith(".xlsx")
subprocess.run(["python", "experiments/putwriting/followup/panel.py", "--workbook", workbook], check=True)


In [ ]:
data = root / "experiments/putwriting/followup/neural/data"
results = root / "experiments/putwriting/followup/neural/results"
data.mkdir(parents=True, exist_ok=True)
(data / "daily_panel.parquet").write_bytes((root / "experiments/putwriting/followup/results/panel.parquet").read_bytes())
env = dict(os.environ, DAILY_H22_DATA_DIR=str(data), DAILY_H22_RESULTS_DIR=str(results), OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
import torch
print("GPU available:", torch.cuda.is_available())
subprocess.run(["python", "experiments/putwriting/followup/neural_src/12_tsmixerx_optuna.py", "--target-trials", "25", "--max-steps", "400", "--timeout-minutes", "480"], env=env, check=True)
subprocess.run(["python", "experiments/putwriting/followup/neural_src/13_neural_rolling.py", "--time-budget-minutes", "90"], env=env, check=True)


In [ ]:
output_file = str(root / "neural_followup_results.zip")
with zipfile.ZipFile(output_file, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in results.rglob("*"):
        if path.is_file(): archive.write(path, path.relative_to(results.parent))
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
